## 1. PureActionState

### Example: Create GENERATE state using PureActionState

```lua
stateDiagram-v2
    INIT --> GENERATE
    GENERATE --> FINAL
```

```mermaid
stateDiagram-v2
    Direction LR
    INIT --> GENERATE
    GENERATE --> FINAL
```


In [ ]:
from gai.asm import AgenticStateMachine


async def generate_action(state):
    from openai import AsyncOpenAI

    client = AsyncOpenAI()

    # Import data from state_bag
    agent_name = state.machine.state_bag.get("name", "Assistant")
    user_message = state.machine.state_bag.get(
        "user_message",
        "If you are seeing this, that means I have forgotten to add a user message. Remind me.",
    )

    from openai import AsyncOpenAI

    client = AsyncOpenAI()

    # Execute

    state.machine.monologue.add_user_message(
        content=f"Your name is {agent_name}. You are a helpful assistant.{agent_name}, {user_message}"
    )

    from gai.messages import message_helper

    chat_messages = state.machine.monologue.list_chat_messages()
    chat_messages = message_helper.shrink_messages(chat_messages)

    response = await client.chat.completions.create(
        model="gpt-4.1",
        messages=chat_messages,
        max_tokens=50,
        stream=True,
    )

    async def streamer():
        content = ""
        async for chunk in response:
            chunk = chunk.choices[0].delta.content
            if isinstance(chunk, str) and chunk:
                content += chunk
                yield chunk
        state.machine.monologue.add_assistant_message(content=content)

    state.machine.state_bag["streamer"] = streamer()


## Step 1: INIT

with AgenticStateMachine.StateMachineBuilder("""
    INIT --> GENERATE
    GENERATE --> FINAL
    """) as builder:
    fsm = builder.build(
        {
            "INIT": {
                "input_data": {},
            },
            "GENERATE": {
                "module_path": "gai.asm.states",
                "class_name": "PureActionState",
                "title": "GENERATE",
                "action": "generate",
                "output_data": ["streamer"],
            },
            "FINAL": {
                "output_data": ["monologue"],
            },
        },
        user_message="Hello, world!",
        generate=generate_action,
    )
fsm.restart()

## Step 2: INIT --> GENERATE

resp = await fsm.run_async()
async for chunk in resp:
    print(chunk, end="", flush=True)
print("\n\n")

## Step 3: GENERATE --> FINAL
await fsm.run_async()

## Step 4: Print the state history
print("State History:")
for state in fsm.state_history:
    print(f"State: {state['state']}")
    print(f"- input: {state['input']}")
    print(f"- output: {state['output']}")
    print("-" * 20)

ValueError: State 'CHAT' is not a registered state.

### Example: Create conditional state using PurePredicateState

In this example, we will demonstrate a **Predicate** state whose sole existence is to decide if the output is **true** or **false**


```mermaid
stateDiagram-v2
INIT --> PREDICATE: next / action
PREDICATE --> TRUE(): next / action / condition_true
    TRUE() --> FINAL: next / action
PREDICATE --> FALSE(): next / action / condition_false
    FALSE() --> FINAL: next / action
```

Ask if the sky is blue

In [ ]:
from gai.asm import AsyncStateMachine

# Define the predicate: ask the LLM if the sky is blue


def sky_is_blue(state) -> bool:
    from gai.llm.openai import OpenAI, boolean_schema

    client = OpenAI(
        client_config={
            "type": "ttt",
            "client_type": "openai",
        }
    )
    response = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[{"role": "user", "content": "The sky is blue."}],
        response_format=boolean_schema,
        max_tokens=100,
        temperature=0.0,
    )
    import json

    content = json.loads(response.choices[0].message.content)  # Parse the JSON response
    return bool(content["result"])


# Define the predicate actions


async def run_if_true(state):
    print("Action: THE ANSWER IS TRUE")


async def run_if_false(state):
    print("Action: THE ANSWER IS FALSE")


with AsyncStateMachine.StateMachineBuilder("""
    INIT --> PREDICATE
    PREDICATE --> TRUE: condition_true
        TRUE --> FINAL 
    PREDICATE --> FALSE: condition_false
        FALSE --> FINAL    
    """) as builder:
    fsm = builder.build(
        {
            "PREDICATE": {
                "module_path": "gai.asm.states",
                "class_name": "PurePredicateState",
                "title": "PREDICATE",
                "predicate": "sky_is_blue",
                "output_data": ["predicate_result"],
                "conditions": ["condition_true", "condition_false"],
            },
            "TRUE": {
                "module_path": "gai.asm.states",
                "class_name": "PureActionState",
                "input_data": {"title": "TRUE"},
                "action": "run_if_true",
            },
            "FALSE": {
                "module_path": "gai.asm.states",
                "class_name": "PureActionState",
                "input_data": {"title": "FALSE"},
                "action": "run_if_false",
            },
        },
        sky_is_blue=sky_is_blue,
        run_if_true=run_if_true,
        run_if_false=run_if_false,
    )


In [ ]:
fsm.restart()
await fsm.run_async()

# state output:
print("Current state is", fsm.state)
print("Result is", fsm.state_bag["predicate_result"])


In [ ]:
await fsm.run_async()

# state output:
print("Current state is", fsm.state)
